1. 离线索引服务
- 只负责：
- 加载文档
- 切块
- embedding
- 写入 Chroma 持久化目录
2. 在线 Agent 服务
- 只负责：
- 接收用户问题
- 判断是否需要查文档
- 如果需要，则调用 query_documents
- 如果需要业务操作，则调用 Asana tools
- 汇总工具结果，生成最终答案
3. RAG 不再“每次固定调用”
- 而是作为一个 Tool，由 Agent 按需调用。
- 这是你现有三种方案里，最接近工业实现的版本

In [ ]:
from app.config import settings


In [ ]:
settings.llm_model

In [ ]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


def load_and_split(directory: str):
    loader = DirectoryLoader(directory)
    documents = loader.load()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=150,
    )
    return splitter.split_documents(documents)

In [ ]:
from sqlalchemy import create_engine, text
from app.config import settings

# pool_pre_ping=True: Every time you take a connection from the connection pool, check whether the connection is still alive to avoid getting invalid connections.
engine = create_engine(settings.postgres_url, pool_pre_ping=True)


def insert_document(source: str, chunk_index: int, content: str, embedding: list[float]) -> None:
    sql = text("""
        INSERT INTO documents (source, chunk_index, content, embedding)
        VALUES (:source, :chunk_index, :content, :embedding)
    """)
    with engine.begin() as conn:
        conn.execute(
            sql,
            {
                "source": source,
                "chunk_index": chunk_index,
                "content": content,
                "embedding": embedding,
            },
        )


def vector_search(query_embedding: list[float], limit: int = 20):
        # <=> is the "cosine distance" operator for pgvector
    # mappings().all()  Convert the result to a "dictionary-like" form and return it instead of a tuple
    sql = text("""
        SELECT id, source, chunk_index, content,
               embedding <=> CAST(:embedding AS vector) AS distance
        FROM documents
        ORDER BY embedding <=> CAST(:embedding AS vector)
        LIMIT :limit
    """)
    with engine.begin() as conn:
        rows = conn.execute(
            sql,
            {"embedding": str(query_embedding), "limit": limit},
        ).mappings().all()
    return rows


In [ ]:
from app.config import settings
from app.rag.embed import get_embedder
from app.rag.ingest import load_and_split
from app.rag.store import insert_document


def main():
    docs = load_and_split(settings.rag_directory)
    embedder = get_embedder()

    text = [doc.page_content for doc in docs]
    vectors = embedder.encode(text, normalize_embeddings= True).tolist()

    for idx, (doc, emb) in enumerate(zip(docs, vectors)):
        source = doc.metadata.get("source","NA")
        insert_document(
            source=source,
            chunk_index=idx,
            content=doc.page_content,
            embedding=emb
        )
    print(f"Indexed {len(docs)} chunks.")

if __name__ == "__main__":
    main()

In [ ]:
# from sqlalchemy import text
# from app.rag.store import engine

# with engine.begin() as conn:
#     count = conn.execute(text("SELECT COUNT(*) FROM documents")).scalar()
#     print("documents count:", count)

#     sample = conn.execute(text("""
#         SELECT id, source, chunk_index, left(content, 200)
#         FROM documents
#         LIMIT 5
#     """)).fetchall()
#     print(sample)

rows = vector_search(query_embedding, 10)

for row in rows:
    print("distance:", row["distance"])
    print("chunk_index:", row["chunk_index"])
    print("content:", row["content"][:300])
    print("-" * 50)


In [ ]:
from sqlalchemy import create_engine, text
from app.config import settings

# pool_pre_ping=True: Every time you take a connection from the connection pool, check whether the connection is still alive to avoid getting invalid connections.
engine = create_engine(settings.postgres_url, pool_pre_ping=True)


def vector_search(query_embedding: list[float], limit: int = 20):
        # <=> is the "cosine distance" operator for pgvector
    # mappings().all()  Convert the result to a "dictionary-like" form and return it instead of a tuple
    sql = text("""
        SELECT id, source, chunk_index, content,
               embedding <=> CAST(:embedding AS vector) AS distance
        FROM documents
        ORDER BY embedding <=> CAST(:embedding AS vector)
        LIMIT :limit
    """)
    with engine.begin() as conn:
        rows = conn.execute(
            sql,
            {"embedding": str(query_embedding), "limit": limit},
        ).mappings().all()
    return rows



embeder = get_embedder()
query_embedding = embeder.encode_query("what is next step.",normalize_embeddings=True).tolist()

rows = vector_search(query_embedding,10)

In [ ]:
from app.rag.embed import get_embedder, get_reranker
from app.rag.store import vector_search
from app.config import settings


def retrieve(question:str, limit = 20 ,topk = 5) -> str:
    embeder = get_embedder()
    reranker = get_reranker()

    query_vec = embeder.encode(question, normalize_embeddings=True).tolist()

    candidates = vector_search(query_vec, limit)
    
    if not candidates:
        return "No relevant documents found."
    
    pairs = [(question, row['content']) for row in candidates]
    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(candidates, scores),
        key = lambda x: x[1],
        reverse= True
    )[:topk]


    return "\n\n".join(
        f"Source: {row['source']}\nContent: {row['content']}"
        for row, _ in ranked
    )



res = retrieve("what is next step.")



In [ ]:
import os
from typing import Annotated, TypedDict
from langchain_core.tools import tool,Tool
from langgraph.prebuilt import tools_condition, ToolNode
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage, AnyMessage
from langgraph.graph import START, StateGraph
from app.tool.rag_tools import query_documents
from app.config import settings
from langchain_openai import ChatOpenAI
from langfuse.langchain import CallbackHandler


os.environ["LANGFUSE_PUBLIC_KEY"] = settings.langfuse_public_key
os.environ["LANGFUSE_SECRET_KEY"] = settings.langfuse_secret_key
os.environ["LANGFUSE_HOST"] = settings.langfuse_host
callbackhandler = CallbackHandler()


tools = [query_documents]

llm = ChatOpenAI(
    model=settings.llm_model,
    temperature=0.1,
    base_url=settings.model_url,
    api_key=settings.token_api_key
)

chat_with_tools = llm.bind_tools(tools)

class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]


def assistant(agent_state: AgentState):
    return {
        'messages': [chat_with_tools.invoke(agent_state['messages'])]
    }


builder = StateGraph(AgentState)
builder.add_node("assistant", assistant )
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "assistant")
builder.add_conditional_edges(
    "assistant",
    tools_condition
)
builder.add_edge('tools', 'assistant')


agent = builder.compile()



In [ ]:
response = agent.invoke(input = {
    'messages': [HumanMessage(content="What is main tasks in our document?")]},
                 config={'callbacks' : [callbackhandler]}
)
response['messages'][-1].content

In [ ]:
from fastapi import APIRouter
from pydantic import BaseModel
from langchain_core.messages import HumanMessage, SystemMessage
from app.agent.agent import agent, callbackhandler


router = APIRouter()

class ChatRequest(BaseModel):
    messages: str

@router.post('/chat')
async def chat(req: ChatRequest):
    messages = [
        SystemMessage(content = """
                            You are an enterprise assistant. Use document retrieval for knowledge questions. Do not expose internal IDs unless explicitly requested."""),
                            HumanMessage(content=req.messages)
    ]

    response = agent.invoke(input = {'messages' : messages}, config = {'callbacks':[callbackhandler]})
    return {"answer": response['messages'][-1].content}

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient

app = FastAPI()
app.include_router(router)

client = TestClient(app)

resp = client.post(
    "/chat",
    json={"messages": "what is the next step?"}
)

print(resp.status_code)
print(resp.json())


In [ ]:
from fastapi import FastAPI
from app.api.routes import router

# Responsible for creating FastAPI applications and registering routes
app = FastAPI(title= 'Enterprise Agent')
app.include_router(router)